# CALCULAR FREQÜÈNCIES CHARS

In [1]:
#from comparative.entities import Dataset

from dataclasses import dataclass
from pathlib import Path

@dataclass
class Dataset:
    dataset_id: str # num o codi curt
    label: str # categoria
    description: str
    fasta_path: Path 
    size_bytes: int
    num_bases: int # només A,C,G,T
    base_counts: dict[str, int] # counts A,C,G,T,N i unknown
    size_header: float # quants caracters te el header
    entropy_acgtn_bits_per_base: float | None # Entropia H0 sobre A,C,G,T,N
    entropy_acgt_bits_per_base: float | None # Entropia H0 sobre A,C,G,T ignorant N
    entropy_seq_all_bits_per_base: float | None # Entropia H0 sobre la seqüència completa
    entropy_seq_all_headers_bits_per_base: float | None # Entropia H0 sobre la seqüència completa i els headers
    contains_n: bool = False

            

In [3]:
import os
from pathlib import Path
from typing import List, Optional
import sys
from pathlib import Path
# print working directory

sys.path.append(str(Path("..").resolve()))
print(f"Working directory: {Path.cwd()}")
from comparative.utils import shannon_entropy_from_counts, compute_dataset_entropies
from comparative.fasta_utils import count_fasta_symbols



Working directory: /home/helen/genomic_benchmark/proves


In [4]:
DATA_DIR = "/home/helen/genomic_benchmark/DATA/data"

datasets = dict()
for fasta_path in sorted(Path(DATA_DIR).rglob("*")):
    if not fasta_path.is_file():
        continue
    print(f"Processing {fasta_path.stem}...")
    datasets[fasta_path.stem] = dict()
    datasets[fasta_path.stem]["category"] = fasta_path.parent.stem
    base_counts = count_fasta_symbols(fasta_path)
    entropies = compute_dataset_entropies(fasta_path, base_counts)
    datasets[fasta_path.stem]["base_counts"] = base_counts
    datasets[fasta_path.stem]["entropies"] = entropies



Processing bacillus_subtilis...
Processing escherichia_coli...
Processing mycobacterium_tuberculosis...
Processing pseudomonas_aeruginosa...
Processing salmonella_typhimurium...
Processing staphylococcus_aureus...
Processing homo_sapiens_chr1...
Processing homo_sapiens_chr21...
Processing homo_sapiens_chr22...
Processing homo_sapiens_chr4...
Processing homo_sapiens_chrX...
Processing homo_sapiens_chrY...
Processing arabidopsis_thaliana...
Processing candidozma_auris...
Processing saccharomyces_cerevisiae...
Processing yarrowia_lipolytica...
Processing PV559941.1...
Processing PV559966.1...
Processing PV559974.1...
Processing PV559982.1...
Processing PV560051.1...
Processing mitochondria...
Processing hiv...
Processing hpv16...
Processing hsv1...
Processing phage_lambda...
Processing phix174...
Processing sars_cov_2...


In [5]:
categories = set()
for dataset in datasets.values():
    categories.add(dataset["category"])

print(f"Categories: {categories}")

Categories: {'eucariota', 'mitochondria', 'virus', 'chromosome', 'bacteria'}


In [6]:
accept = set("ACGTN")
total_unknown = dict()
total_known = dict()

for cat in categories:
    cat_datasets = {
        k: v for k, v in datasets.items()
        if v["category"] == cat
    }
    
    for dataset in cat_datasets.values():
        for k, v in dataset["base_counts"].get("unknown", {}).items():
            total_unknown[k] = total_unknown.get(k, 0) + v
    
        for i in accept:
            total_known[i] = total_known.get(i, 0) + dataset["base_counts"].get(i, 0)

print("Total unknown symbols across all datasets:")

total = sum(total_unknown.values()) + sum(total_known.values())
for k, v in total_unknown.items():
    print(f"{k}: {v:,} ({v/total:.4%})", end = "\t")
print("\n\nTotal known symbols across all datasets:")
for k, v in total_known.items():
    print(f"{k}: {v:,} ({v/total:.4%})", end = "\t")


Total unknown symbols across all datasets:
Y: 88 (0.0000%)	W: 127 (0.0000%)	M: 79 (0.0000%)	K: 53 (0.0000%)	S: 32 (0.0000%)	R: 42 (0.0000%)	

Total known symbols across all datasets:
N: 69,341,536 (8.5026%)	T: 220,988,960 (27.0974%)	C: 152,162,504 (18.6580%)	A: 220,512,437 (27.0389%)	G: 152,530,952 (18.7031%)	

In [7]:
accept = set("ACGTN")
total_unknown = dict()
total_known = dict()

for cat in categories:
    cat_datasets = {
        k: v for k, v in datasets.items()
        if v["category"] == cat
    }
    total_unknown[cat] = dict()
    total_known[cat] = dict()
    for dataset in cat_datasets.values():
        for k, v in dataset["base_counts"].get("unknown", {}).items():
            total_unknown[cat][k] = total_unknown[cat].get(k, 0) + v
    
        for i in accept:
            total_known[cat][i] = dataset["base_counts"].get(i, 0) + v

print("Total unknown symbols across all datasets:")

for cat in categories:
    print(f"{cat}:")
    total = sum(total_unknown[cat].values()) + sum(total_known[cat].values())
    for k, v in total_unknown[cat].items():
        print(f"{k}: {v} ({v/total:.4%})", end = "\t")
    print("\n") if total_unknown[cat] else None
    for k, v in total_known[cat].items():
        print(f"{k}: {v} ({v/total:.4%})", end = "\t")
    print("\n")

Total unknown symbols across all datasets:
eucariota:
Y: 82 (0.0020%)	W: 124 (0.0030%)	M: 76 (0.0018%)	K: 53 (0.0013%)	S: 30 (0.0007%)	R: 36 (0.0009%)	

N: 36 (0.0009%)	T: 1063687 (25.3312%)	C: 1037209 (24.7007%)	A: 1063450 (25.3256%)	G: 1034332 (24.6321%)	

mitochondria:
N: 37 (0.2209%)	T: 4130 (24.6582%)	C: 5217 (31.1481%)	A: 5160 (30.8078%)	G: 2205 (13.1650%)	

virus:
N: 36 (0.1197%)	T: 9630 (32.0114%)	C: 5528 (18.3758%)	A: 8990 (29.8840%)	G: 5899 (19.6091%)	

chromosome:
M: 3 (0.0000%)	R: 6 (0.0000%)	W: 3 (0.0000%)	Y: 6 (0.0000%)	S: 2 (0.0000%)	

N: 30812368 (53.8420%)	T: 7956169 (13.9027%)	C: 5285790 (9.2365%)	A: 7886193 (13.7804%)	G: 5286895 (9.2384%)	

bacteria:
N: 2 (0.0001%)	T: 955316 (33.8601%)	C: 465833 (16.5109%)	A: 938714 (33.2716%)	G: 461501 (16.3574%)	

